# Midterm Part A - thin Colab launcher
Sparse-checks-out only `midterm_a/` from your fork, installs it and runs `train.py`. All logic lives in the repo. Runtime -> Change runtime type -> **T4 GPU**. After a disconnect re-run the setup cell and the same command: runs resume from `last.ckpt`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
FORK_URL = "https://github.com/rafihi/representation-learning-course.git"
WORK = "/content/drive/MyDrive/ssl_midterm_a"      # checkpoints, results and the dataset cache live on Drive
!mkdir -p {WORK}/runs {WORK}/results {WORK}/data_cache
!rm -rf /content/repo && git clone --filter=blob:none --sparse {FORK_URL} /content/repo
%cd /content/repo
!git sparse-checkout set midterm_a
%cd midterm_a
!ln -s {WORK}/runs runs && ln -s {WORK}/results results && ln -s {WORK}/data_cache data_cache
!pip install -q -r requirements.txt

## 0. Sanity checks (no downloads)

In [ ]:
!SSL_FAKE_DATA=1 pytest -q tests 2>&1 | tail -3

In [ ]:
!python train.py --config configs/a1_tune_dae.yaml --smoke --set out_dir=/tmp/smoke_a1 trainer.progress=false 2>&1 | tail -5

## 1. A2 - tuning (proxy) + reconstruction-vs-kNN figure

In [ ]:
!python train.py --config configs/a1_tune_dae.yaml

Open `results/a1_tune_dae/results.csv`, pick the trial with the best `knn_top1`, copy its `model.kind / level / lr / weight_decay` into `configs/a2_dae_final.yaml` (commit + push, re-run the setup cell), then:

In [ ]:
!python train.py --config configs/a2_dae_final.yaml

In [ ]:
!python train.py --config configs/a3_vanilla_ae_final.yaml

## 2. A3 - frozen protocol vs. baselines (k-NN + linear probe)

In [ ]:
!python train.py --config configs/a4_eval.yaml

## 3. A3/A4 - fine-tuning and label efficiency

In [ ]:
!python train.py --config configs/a5_finetune.yaml

## 4. A4 - corruption curves

In [ ]:
!python train.py --config configs/a6_robustness.yaml

## 5. A5 - t-SNE, retrieval grid, per-class accuracy

In [ ]:
!python train.py --config configs/a7_interpret.yaml

## Final numbers on the TEST split (only once, after every decision is made)

In [ ]:
!python train.py --config configs/a4_eval.yaml --set split=test out_dir=results/final_a4_test
!python train.py --config configs/a5_finetune.yaml --set split=test out_dir=results/final_a5_test

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs